In [1]:
import os
import numpy as np
import pandas as pd
from MDAnalysis.analysis.distances import distance_array
import MDAnalysis as mda

# Load trajectories (.xtc) and topology file (.tpr)
trajfiles2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1_f.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep2_f.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep3_f.xtc",
            )
topfile2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1.tpr")
trajfiles3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1_f.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep2_f.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep3_f.xtc",
            )
topfile3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1.tpr")
bdn1 = mda.Universe(topfile2, trajfiles2[0])
bdn2 = mda.Universe(topfile2, trajfiles2[1])
bdn3 = mda.Universe(topfile2, trajfiles2[2])
bdn = mda.Universe(topfile2, trajfiles2)

bdq1 = mda.Universe(topfile3, trajfiles3[0])
bdq2 = mda.Universe(topfile3, trajfiles3[1])
bdq3 = mda.Universe(topfile3, trajfiles3[2])
bdq = mda.Universe(topfile3, trajfiles3)
# =========================================================
# BR CONTACTS FUNCTION
# =========================================================

def bromine_contacts_timeseries(u, ligand_resid, pocket_resids, cutoff=4.0):

    ligand = u.select_atoms(
        f"resid {ligand_resid} and name BR"
    )

    pocket = u.select_atoms(
        f"protein and resid {' '.join(map(str, pocket_resids))} and not name H*"
    )

    if len(ligand) == 0:
        raise ValueError(f"No Br atoms found in ligand {ligand_resid}")

    if len(pocket) == 0:
        raise ValueError(f"No pocket atoms found for residues {pocket_resids}")

    contacts = []

    for ts in u.trajectory:

        d = distance_array(
            ligand.positions,
            pocket.positions,
            box=u.dimensions
        )

        contacts.append(np.sum(d < cutoff))

    return np.array(contacts)


# =========================================================
# MAIN PIPELINE (1-to-1 ligand ↔ pocket mapping)
# =========================================================

def run_br_contact_analysis(
    systems,
    pairs,
    outdir,
    cutoff=4.0
):

    os.makedirs(outdir, exist_ok=True)

    # -----------------------------------------------------
    # LOOP SYSTEMS
    # -----------------------------------------------------

    for system_name, replicas in systems.items():

        system_dir = os.path.join(outdir, system_name)
        os.makedirs(system_dir, exist_ok=True)

        # -------------------------------------------------
        # LOOP DEFINED LIGAND–POCKET PAIRS ONLY
        # -------------------------------------------------

        for lig_name, (pocket_name, lig_resid, pocket_resids) in pairs.items():

            print(f"{system_name} | {lig_name} ↔ {pocket_name}")

            lig_dir = os.path.join(system_dir, lig_name)
            os.makedirs(lig_dir, exist_ok=True)

            all_replicas = []

            # -------------------------------------------------
            # LOOP REPLICAS
            # -------------------------------------------------

            for rep_idx, u in enumerate(replicas, start=1):

                contacts = bromine_contacts_timeseries(
                    u,
                    lig_resid,
                    pocket_resids,
                    cutoff=cutoff
                )

                df = pd.DataFrame({

                    "System": system_name,
                    "Ligand": lig_name,
                    "LigandResid": lig_resid,
                    "Pocket": pocket_name,
                    "Replica": rep_idx,
                    "Frame": np.arange(len(contacts)),
                    "BrContacts": contacts

                })

                all_replicas.append(df)

            combined = pd.concat(all_replicas, ignore_index=True)

            outpath = os.path.join(
                lig_dir,
                f"{pocket_name}_br_contacts_timeseries_bis.csv"
            )

            combined.to_csv(outpath, index=False)

            print(f"Saved -> {outpath}")


# =========================================================
# INPUT EXAMPLE
# =========================================================

systems = {
    "BDN": [bdn1, bdn2, bdn3],
    "BDQ": [bdq1, bdq2, bdq3]
}

pairs = {
    "lig1064": ("lagging", 1064, [298, 299, 301, 302, 305, 265, 377, 380, 381, 384, 160, 162, 163]),
    "lig1065": ("leading", 1065, [207, 208, 211, 541, 542, 544, 545, 548, 508, 620, 623, 624, 627]),
    "lig1066": ("site3", 1066, [622, 623, 625, 626, 629, 589, 701, 704, 705, 708]),
    "lig1067": ("site4", 1067, [703, 704, 706, 707, 710, 670, 782, 785, 786, 789]),
    "lig1068": ("site5", 1068, [784, 785, 787, 788, 791, 751, 863, 866, 867, 870]),
    "lig1069": ("site6", 1069, [865, 866, 868, 869, 872, 832, 944, 947, 948, 951]),
    "lig1070": ("site7", 1070, [946, 947, 949, 950, 953, 913, 296, 299, 300, 303]),
}

# =========================================================
# RUN
# =========================================================

run_br_contact_analysis(
    systems=systems,
    pairs=pairs,
    outdir="/scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts",
    cutoff=5.0
)

BDN | lig1064 ↔ lagging


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDN/lig1064/lagging_br_contacts_timeseries_bis.csv
BDN | lig1065 ↔ leading


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDN/lig1065/leading_br_contacts_timeseries_bis.csv
BDN | lig1066 ↔ site3


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDN/lig1066/site3_br_contacts_timeseries_bis.csv
BDN | lig1067 ↔ site4


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDN/lig1067/site4_br_contacts_timeseries_bis.csv
BDN | lig1068 ↔ site5


...

Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDQ/lig1066/site3_br_contacts_timeseries_bis.csv
BDQ | lig1067 ↔ site4


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDQ/lig1067/site4_br_contacts_timeseries_bis.csv
BDQ | lig1068 ↔ site5


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDQ/lig1068/site5_br_contacts_timeseries_bis.csv
BDQ | lig1069 ↔ site6


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDQ/lig1069/site6_br_contacts_timeseries_bis.csv
BDQ | lig1070 ↔ site7


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/br_contacts/BDQ/lig1070/site7_br_contacts_timeseries_bis.csv
